# Two-operation rules, "sequence of rigid transformations" prompt — 3 models × 1/3/5/7/9 examples

**Prompt header** (notebooks 48-50 header plus one sentence):

> Each example below shows a 6x6 grid of digits (Input) and the grid obtained by applying the same hidden
> transformation to it (Output). The transformation is the same for every example. The hidden transformation is a
> sequence of one or more rigid transformations (distance-preserving maps) of the grid, where the grid wraps around
> at its edges.

No operation is named, and the number of steps is not given.

**Two experiments per task:**
* `answer`: give the transformed grid only.
* `steps`: "break the transformation down into the steps it is made of", one `<step>` per step, then the
  final grid. No intermediate grids are requested.

**Tasks.** The 6 two-operation rules AB, AC, BA, BC, CA, CB (A = shift rows right, B = rotate 90° clockwise,
C = mirror left-right); 20 tasks each. Examples are **nested**: each task has 9 examples, and the run with n examples
uses the first n, with the same query grid. The first example alone already determines the rule uniquely.

**Models:** `openai/gpt-6-luna`, `openai/gpt-6.2-astra`, `meta/muse-glimmer-30b` (each with its default reasoning).
**Size:** 120 tasks × 5 example counts × 2 experiments × 3 models = **3,600 calls**.

Cell 4 = dry run (no cost). Cell 5 = pilot: 2 calls per model (checks model IDs and output format).
Cell 6 = full run (resumable). Cell 7 = tables, figure and zip.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- 3. Configuration ---------------------------------------------------------------------------------------------
MODELS = ["openai/gpt-6-luna", "openai/gpt-6.2-astra", "meta/muse-glimmer-30b"]
DEMO_COUNTS = [1, 3, 5, 7, 9]
N_PER_WORD = 20
TASK_SEED = 0
EXPERIMENTS = ["answer", "steps"]
STYLE = "steps"                  # header: sequence of rigid transformations, wrap-around grid
REASONING = None                 # None = each model's default
MAX_TOKENS = None
WORKERS = 32
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if IN_COLAB and not os.environ.get("OPENROUTER_API_KEY"):
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")      # never printed
print("OpenRouter key present:", bool(os.environ.get("OPENROUTER_API_KEY")))
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 250)
from decompositionality import icl_api as IA
_tag = "default" if not REASONING else "_".join(f"{k}{v}" for k, v in REASONING.items())
OUT_ROOT = pathlib.Path(RESULTS_ROOT) / "icl_api" / f"k2_rigid_steps_n{N_PER_WORD}_seed{TASK_SEED}_{_tag}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
path = lambda m, n: OUT_ROOT / m.replace("/", "_") / f"demos{n}.jsonl"
print("results ->", OUT_ROOT)


In [ ]:
# --- 4. Dry run: build the tasks and show both prompts (no API calls) -----------------------------------------------
tasks = IA.make_nested_tasks(IA.K2_WORDS, n_per_word=N_PER_WORD, max_demos=max(DEMO_COUNTS), seed=TASK_SEED)
(OUT_ROOT / "tasks.json").write_text(json.dumps(tasks))
print(len(tasks), "tasks")
for e in EXPERIMENTS:
    print(f"\n==================== experiment '{e}' (task {tasks[0]['task_id']}, 1 example) ====================")
    print(IA.build_prompt(IA.with_demos(tasks, 1)[0], e, hint=False, style=STYLE))
print("\ncalls planned:", len(tasks) * len(DEMO_COUNTS) * len(EXPERIMENTS) * len(MODELS))


In [ ]:
# --- 5. Pilot: 2 calls per model (3 examples) ------------------------------------------------------------------------
for m in MODELS:
    IA.run(IA.with_demos(tasks, 3)[:1], path(m, 3), m, experiments=EXPERIMENTS, hint=False, style=STYLE,
           reasoning=REASONING, max_tokens=MAX_TOKENS, workers=WORKERS)
    for line in path(m, 3).read_text().splitlines()[-2:]:
        r = json.loads(line)
        print(f"--- {m} / {r['experiment']}  error={r['error']}\n{(r['content'] or '')[:1500]}\n")


In [ ]:
# --- 6. Full run (resumable: calls already saved are skipped) --------------------------------------------------------
for m in MODELS:
    for n in DEMO_COUNTS:
        print(f"=== {m}, {n} examples")
        IA.run(IA.with_demos(tasks, n), path(m, n), m, experiments=EXPERIMENTS, hint=False, style=STYLE,
               reasoning=REASONING, max_tokens=MAX_TOKENS, workers=WORKERS)


In [ ]:
# --- 7. Tables and figure --------------------------------------------------------------------------------------------
df = pd.concat([IA.score(IA.with_demos(tasks, n), path(m, n)).assign(model=m, n_demos=n)
                for m in MODELS for n in DEMO_COUNTS if path(m, n).exists()], ignore_index=True)
print("Accuracy by model, experiment and number of examples:")
display(df.pivot_table(index="n_demos", columns=["model", "experiment"], values="correct").round(3))
print("\nAccuracy by rule (all example counts):")
display(df.pivot_table(index="word", columns=["model", "experiment"], values="correct").round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, e in zip(axes, EXPERIMENTS):
    for m in MODELS:
        q = df[(df.model == m) & (df.experiment == e)].groupby("n_demos").correct.mean()
        ax.plot(q.index, q.values, marker="o", lw=2, label=m)
    ax.set(title=e, xlabel="number of examples", xticks=DEMO_COUNTS, ylim=(0, 1.02)); ax.grid(alpha=.3)
axes[0].set_ylabel("accuracy"); axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(OUT_ROOT / "accuracy.png", dpi=150); plt.show()
print("\nError types:")
display(df.groupby(["model", "experiment"]).error_type.value_counts(normalize=True).unstack(fill_value=0).round(3))
s = df[df.experiment == "steps"]
print("\nNumber of steps the model writes (rows: model, columns: number of steps):")
display(pd.crosstab([s.model, s.word], s.n_steps))
print("\nExample decompositions (3 examples, first 2 tasks per rule):")
display(s[s.n_demos == 3].groupby(["model", "word"]).head(2)[["model", "word", "correct", "n_steps", "steps_text"]])
print("\nUsage per call:")
display(df[["prompt_tokens", "completion_tokens", "reasoning_tokens", "cost_usd"]].apply(pd.to_numeric, errors="coerce")
        .groupby([df.model, df.experiment]).mean().round(4))
print("total cost by model ($):", df.assign(c=pd.to_numeric(df.cost_usd, errors="coerce")).groupby("model").c.sum().round(2).to_dict())
df.to_csv(OUT_ROOT / "scores.csv", index=False)
print("saved to", OUT_ROOT)
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive(str(OUT_ROOT), "zip", OUT_ROOT)
    files.download(str(OUT_ROOT) + ".zip")
